# 05 - QpiAI Quantum / QCloud local simulator experiment

**Purpose:** Reproduce the reported -13.07 QAOA energy and apply the same QUBO formulation to the Phase 2 Wave-0 selection; chart the sampled distribution.

**Inputs:** qpiai-opt[quantum], qpiai-quantum; device QpiAI-QSV-Local (in-process statevector)

**Outputs:** reproduction result, counts chart (feasible vs infeasible, optimum marked), P(optimum) vs uniform

**Status labels:** SIMULATOR only. No QPU, no platform mode. No quantum advantage is claimed.

Context: the demo bank in `demo_bank_app/` is SYNTHETIC. Charts are also saved to `scratch/derived/member3/figs/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
FIGS = DERIVED / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
PY = sys.executable

def ensure(rel_path, script, *args):
    """Run a project script once if its output file is missing (keeps notebooks runnable in a fresh checkout)."""
    target = ROOT / rel_path
    if not target.is_file():
        print(f"{rel_path} missing -> running {script} ...")
        subprocess.run([PY, str(ROOT / "scripts" / script), *args], cwd=ROOT, check=True)
    return target

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS / f"{name}.png", dpi=120); plt.show()
# This notebook reads no credentials. If one is ever needed, read it from an environment variable at runtime and
# never display it.

In [ ]:
from cryptoshift.optimization.member3_qpiai import reproduce_reported_experiment
r = reproduce_reported_experiment()
pd.Series({k: r[k] for k in ('local_qsv_simulator_status','best_simulator_wave0_selection','best_simulator_wave0_energy','reproduced')}).to_frame('reproduction')

In [ ]:
from cryptoshift.planning.planner import build_problem
from cryptoshift.planning.dependency_graph import constraint_digraph
from cryptoshift.optimization.member3_qpiai import run_wave0_qaoa, wave0_energy
p = build_problem(); c = p['config']
d = constraint_digraph(p['graph'], c['enforced_edge_provenance'])
cands = [{'id': s, 'reward': v['risk']['score'], 'minutes': v['duration_minutes']} for s, v in sorted(p['services'].items()) if v['verification_ready_wave0']]
ids = {x['id'] for x in cands}
q = run_wave0_qaoa(cands, sorted((u, v) for u, v in d.edges if u in ids and v in ids), c['constraints']['max_services_per_wave'], c['constraints']['max_minutes_per_wave'])
print(q['label']); print('variables (bit order little-endian):', q['formulation']['variables'])
pd.Series({k: q[k] for k in ('exact_optimum','optimum_probability','uniform_probability','feasible_shot_fraction','distinct_bitstrings_sampled','circuit')}).to_frame('Phase 2 Wave-0 QAOA')

In [ ]:
names = q['formulation']['variables']
terms = {k: q['formulation'][k] for k in ('penalty','capacity','deps','pairs_over_window','rewards')}
shots = {x['bits']: x['count'] for x in q['samples']}
rows = []
for bits in (format(i, f'0{len(names)}b') for i in range(2 ** len(names))):
    sel = {names[i]: int(bits[len(bits)-1-i]) for i in range(len(names))}
    e, ok, _ = wave0_energy(sel, terms)
    rows.append({'bits': bits, 'selection': '+'.join(k[0].upper() for k, v in sel.items() if v) or 'none', 'energy': e, 'feasible': ok, 'shots': shots.get(bits, 0)})
df = pd.DataFrame(rows).sort_values('energy').reset_index(drop=True)
print('Legend: ' + ', '.join(f'{n[0].upper()} = {n}' for n in names) + ' | bit strings are little-endian (rightmost bit = first variable)')
labels = df['selection']
feas_col = ['#2a9d8f' if f else '#e76f51' for f in df['feasible']]
fig, ax = plt.subplots(3, 1, figsize=(14, 11))
ax[0].bar(labels, df['energy'], color=feas_col)
ax[0].set_title('Exact QUBO energy of every Wave-0 selection, best first (green = feasible, red = violates constraints)'); ax[0].set_ylabel('energy (lower is better)'); ax[0].tick_params(axis='x', labelsize=9)
ax[1].bar(labels, df['shots'], color=feas_col)
ax[1].axhline(q['shots'] / 2 ** len(names), color='black', ls='--', label='uniform expectation')
ax[1].annotate('exact optimum', (0, df.loc[0, 'shots']), xytext=(1.5, df['shots'].max()), arrowprops={'arrowstyle': '->'})
ax[1].set_title(f"QAOA samples on QpiAI-QSV-Local ({q['shots']} shots, same order as above)"); ax[1].set_ylabel('shots'); ax[1].legend(); ax[1].tick_params(axis='x', labelsize=9)
ax[2].bar(['QAOA P(optimum)', 'uniform 1/16', 'feasible shot fraction'], [q['optimum_probability'], q['uniform_probability'], q['feasible_shot_fraction']], color=['#457b9d', '#9e9e9e', '#2a9d8f'])
for i, v in enumerate([q['optimum_probability'], q['uniform_probability'], q['feasible_shot_fraction']]): ax[2].text(i, v, f'{v:.3f}', ha='center', va='bottom')
ax[2].set_title('Summary probabilities (SIMULATOR)'); ax[2].set_ylim(0, 1)
savefig('05_qaoa_energy_and_samples')
df

**Plain language.** The left chart shows the exact energy of all 16 possible Wave-0 choices; the lowest green bar is the true best choice (it equals the deterministic baseline's Wave 0). The right chart shows how often the one-layer, fixed-angle QAOA circuit on the simulator actually lands on that best choice compared with random guessing: only slightly more often. The old experiment's 'best feasible sample' metric finds the optimum for almost any sampler on 4 variables, and its bit decoding was reversed (bit order measured: little-endian).